# PyTorch Autograd Reference

**`torch.autograd`** is PyTorch’s built-in **automatic differentiation engine** that powers neural network training. It tracks operations on tensors to dynamically build a computational graph, allowing you to compute gradients automatically via the chain rule.

---

### Core Components

* **`requires_grad=True`**: This flag alerts PyTorch to track every operation on the specified tensor. Any tensor derived from it inherits this setting.
* **`grad_fn`**: Tensors created by operations store a reference to the specific mathematical function (e.g., `AddBackward0`) needed to compute the gradient.
* **`backward()`**: Calling this method on an output tensor (usually a scalar loss) computes the gradients of that tensor with respect to all tracked leaf tensors.
* **`grad`**: Once `backward()` runs, the computed partial derivatives are accumulated and stored in each leaf tensor's `.grad` attribute.

---

### How the Computational Graph Works

Conceptually, autograd maintains a **Directed Acyclic Graph (DAG)** of your calculations.

1. **Forward Pass**: PyTorch computes the output values and simultaneously builds the execution graph, setting up the backward functions.
2. **Backward Pass**: When `.backward()` is called, autograd traverses the DAG in reverse, calculates local gradients using the `grad_fn` references, and applies the chain rule.
3. **Dynamic Graph**: PyTorch builds this graph **dynamically at runtime** from scratch after every iteration. This allows you to use standard Python control flow statements like loops and `if` conditions in your model architecture.

---

In [7]:
import torch

In [8]:
x = torch.tensor(4.0,requires_grad=True) # We must make sure that the requires_grad is true to calculate the gradients.

In [9]:
y = x ** 2

In [15]:
x

tensor(4., requires_grad=True)

In [17]:
y # The gradient function is PowBackward because we there is a relation of power between y and x which is in the computational graph stored by torch

tensor(16., grad_fn=<PowBackward0>)

In [10]:
y.backward() # Initiate the backward pass , run only once.

In [12]:
# Now to check the gradient value, we use the grad attribute of x
x.grad

tensor(8.)

In [13]:
# It is that simple

In [14]:
# Now let us take a more complex example

In [18]:
x = torch.tensor(6.0,requires_grad=True)
y = x**2
z = torch.cos(y)

In [19]:
x

tensor(6., requires_grad=True)

In [20]:
y

tensor(36., grad_fn=<PowBackward0>)

In [22]:
z # Cosbackward as expected.

tensor(-0.1280, grad_fn=<CosBackward0>)

In [23]:
z.backward()

In [26]:
x.grad # The gradient value from z > y > x through chain rule.

tensor(11.9013)

In [27]:
# In a DAG(Direct Acyclic Graph) Input tensors are leaf and output tensors are root.

### 1. Classification from Scratch
Goal: Update weight `w` for a single input `x` and target `y` (0 or 1).

**Equations:**
- $z = wx + b$
- $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$
- $Loss = -(y \log(\hat{y}) + (1-y) \log(1-\hat{y}))$

**Gradients:**
- $\frac{\partial Loss}{\partial \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1 - \hat{y})}$
- $\frac{\partial \hat{y}}{\partial z} = \hat{y}(1 - \hat{y})$
- $\frac{\partial Loss}{\partial z} = \hat{y} - y$ (Simplified via chain rule)

In [31]:
import math

# Data & Params
x, y = 1.0, 1.0
w, b = 0.5, 0.0
lr = 0.1

# --- Step 1: Forward ---
z = w * x + b
y_hat = 1 / (1 + math.exp(-z))
loss = -(y * math.log(y_hat) + (1 - y) * math.log(1 - y_hat))

# --- Step 2: Backward ---
# Gradient of loss w.r.t z is (y_hat - y)
dz = y_hat - y
dw = dz * x
db = dz

# --- Step 3: Update ---
w -= lr * dw
b -= lr * db

print(f"Manual - Loss: {loss:.4f}, Prediction: {y_hat:.4f}")
print(f"Manual - Updated w: {w:.4f}, b: {b:.4f}")

Manual - Loss: 0.4741, Prediction: 0.6225
Manual - Updated w: 0.5378, b: 0.0378


As you can see, for $x^2$ it's easy. But for a neural network with millions of parameters and complex activation functions, deriving these derivatives by hand would be nearly impossible. That's why `autograd` is so powerful!

### 2. Classification with PyTorch
Now we do the same using `nn.Module` and `BCELoss`.

In [34]:
import torch
import torch.nn as nn

# Setup
x_tensor = torch.tensor([[1.0]])
y_tensor = torch.tensor([[1.0]])

# Model mimicking z = wx + b followed by sigmoid
model = nn.Sequential( # This is just model builidng which we will learn a bit later
    nn.Linear(1, 1),
    nn.Sigmoid()
)

# Initialize with same weights for comparison
with torch.no_grad(): # Inital without any gradient value
    model[0].weight.fill_(0.5)
    model[0].bias.fill_(0.0)

criterion = nn.BCELoss()

# Forward
pred = model(x_tensor)
loss_pt = criterion(pred, y_tensor)

# Backward
loss_pt.backward()

# Update
with torch.no_grad(): # This helps to stop the gradient tracking
# Use no to stop tracking gradients when prediction
    for p in model.parameters():
        p -= 0.1 * p.grad
    model.zero_grad() # what this does is it clears the gradient values.
    # Withou this method the gradients will keep on accumulating.

print(f"PyTorch - Loss: {loss_pt.item():.4f}, Prediction: {pred.item():.4f}")
print(f"PyTorch - Updated w: {model[0].weight.item():.4f}, b: {model[0].bias.item():.4f}")

PyTorch - Loss: 0.4741, Prediction: 0.6225
PyTorch - Updated w: 0.5378, b: 0.0378
